# Wheel encoder based odometry

"Odometry" is the problem of "measuring the path", or evolution of the pose in time, of the robot.

We can solve the odometry problem by using the measurements from wheel encoders, and a so called "dead-reckoning" model, to estimate the evolution of the pose in time through an iterative procedure, such that:

### A comment on notation:
In the slides we use 'S' to denote the magnitude however in these equations 'S' is 'A'. The rest of the notation should be the same apart from the time steps denoted as 'k'.

<div style="text-align:center;">
    <img src="https://raw.githubusercontent.com/duckietown/duckietown-lx/0b59215ef7ff4ce7ae608b9546db3ef9a840d9d0/modcon/assets/images/odometry/odometry-1.png" width="500" alt="odometry-1">
</div>





$$ x_{k+1} = x_k + \Delta x_k $$
$$ y_{k+1} = y_k + \Delta y_k $$
$$ \theta_{k+1} = \theta_k + \Delta \theta_k $$

Where initial conditions ($x_0$, $y_0$, $\theta_0$) are assumed to be known. The increments can be calculated by:

1. **Determining the rotation of each wheel through the wheel encoder mesurements**

$$\Delta \phi_k = N_k \cdot \alpha$$

where $N_k$ is the number of pulses, or "ticks", measured from the encoders in the $k-th$ time interval, $\alpha = \frac{2 \pi}{N_{tot}}$ is the rotation per tick, and $N_{tot}$ the total number of ticks per revolution ($N_{tot} = 135$ for the wheel encoders we will be using). This relation is evaluated for each wheel, yielding $\Delta \phi_{l,k}$ and $\Delta \phi_{r,k}$ for the left and right wheels respectively.

2. **Deriving the total distance travelled by each wheel**

<p style="text-align:center;"><img src="https://raw.githubusercontent.com/duckietown/duckietown-lx/0b59215ef7ff4ce7ae608b9546db3ef9a840d9d0/modcon/assets/images/odometry/odometry-d.png" width="250" alt="odometry-d"></p>

Assuming the wheel radii are the same (equal to $R$) for both wheels, the distance travelled by each wheel is given by:

$$ d_{l/r, k} = R \cdot \Delta \phi_{l/r,k}$$

3. **Finding the rotation and distance travelled by the robot (frame)**  

![](https://drive.google.com/uc?export=view&id=1jxmiKafx5V_v0iy3oi1lx5NJ8AioMgtC)

Under the assumption of no slipping of the robot wheels, we can derive the distance travelled by the origin of the robot frame (point $A$) and the rotation of the robot $\Delta \theta$:

$$ d_{A, k} = \frac{d_{r,k} + d_{l,k}}{2} $$
$$ \Delta \theta_{k} = \frac{d_{r,k} - d_{l,k}}{L}$$

4. **Expressing the robot motion in the world reference frame**

<p style="text-align:center;"><img src="https://raw.githubusercontent.com/duckietown/duckietown-lx/0b59215ef7ff4ce7ae608b9546db3ef9a840d9d0/modcon/assets/images/odometry/odometry-3.png" width="250" alt="odometry-3"></p>

Finally, we can express the estimated motion in the world reference frame and find:

$$ \Delta x_k = d_{A, k} \cos(\theta_k+\frac{\Delta\theta}{2}) $$
$$ \Delta y_k = d_{A, k} \sin(\theta_k +\frac{\Delta\theta}{2})$$


# 🚙 💻 Let's get started!

In this activity you will write a function that produces an estimate of the pose of the Duckiebot, given mesurements from the wheel encoders and an initial position:

In [ ]:
import numpy as np

x0 = y0 = 0.0 # meters
theta0 = np.deg2rad(0) # radians

print(x0,y0, theta0)

## 1. Determining the rotation of each wheel through the wheel encoder mesurements

We have seen how to read wheel encoder data in the [wheel encoder tutorial](https://colab.research.google.com/drive/1ZaNyQSC2yl0p5eI9wu5XqCchTGatx15u?usp=sharing). We can now use this data to measure the rotation of each wheel.

### Wheel encoder calibration factor

Remember that there are 135 ticks per revolution on the wheel encoders we are using.

In [ ]:
# Write the correct expressions
import numpy as np

N_tot = 135 # total number of ticks per revolution
alpha =  # wheel rotation per tick in radians

print(f"The angular resolution of our encoders is: {np.rad2deg(alpha)} degrees")

Assume that at the current update the left and right motor encoders have produced the following measurements:

In [ ]:
# Feel free to play with the numbers to get an idea of the expected outcome

ticks_left = 135
prev_tick_left = 0

ticks_right = 0
prev_tick_right = 0

How much did each wheel rotate?

In [ ]:
# How much would the wheels rotate with the above tick measurements?

# Repetita iuvant: don't confuse degrees and radians when expressing angles
# Machines always use radians, humans make sense of degrees better.
# Mixing these up is a very very common source of error!

delta_ticks_left =  # delta ticks of left wheel
delta_ticks_right = # delta ticks of right wheel
rotation_wheel_left =  # total rotation of left wheel
rotation_wheel_right =  # total rotation of right wheel

print(f"The left wheel rotated: {np.rad2deg(rotation_wheel_left)} degrees")
print(f"The right wheel rotated: {np.rad2deg(rotation_wheel_right)} degrees")

## 2. 🚙 💻 Evaluate distance travelled by each wheel

Now let's calculate the distance travelled by each wheel. It depends on the wheel radii. We need to determine them! We could use advanced odometry calibration procedures, but let's take it a step at the time.

If you have a robot, take a ruler and measure your wheel radii (let's assume they are the same):

In [ ]:
# What is the radius of your wheels (assuming they are identical)?

R = 0.0318 # insert value measured by ruler (meters)

Note: the default value used in simulation and on the robot is $R = 0.0318 \text{m}$.

In [ ]:
# What is the distance travelled by each wheel?

d_left =
d_right =

print(f"The left wheel travelled: {d_left} meters")
print(f"The right wheel rotated: {d_right} meters")

### 🚙 (Optional) Save your new value of `R` (You shouldn't have to do this unless your wheels are way out of wack)


If you have a Duckiebot, let's make sure it remembers its new wheel radius!

Power you Duckiebot on, make sure it is connected to the network and you can ping it, then open a terminal **on your computer** and type:

    dts start_gui_tools ROBOTNAME
    
    rosparam set /ROBOTNAME/kinematics_node/radius R-value
    
where `R-value` is the value of the wheel radius you measured (expressed in meters). You can then save it with:

    rosservice call /ROBOTNAME/kinematics_node/save_calibration
    
and finally verify that it has been saved by opening the `ROBOTNAME.yaml` file in your Dashboard > File Manager > Calibrations > Kinematics page.

You can keep the terminal you just used open, so we can save the baseline measurement too. Let's keep going!

## 3. 🚙 💻 Find the rotation and distance travelled by the Duckiebot

If you have previoulsy set your robot's gain so that the wheels do not slip, the travelled distance of point $A$ (origin of the robot frame) will be given by the average of the distances travelled by the wheels:

In [ ]:
# How much has the robot travelled?

d_A =  # robot distance travelled in robot frame (meters)

print(f"The robot has travelled: {d_A} meters")

To calculate the rotation of the robot we need to measure the baseline too - or the distance between the center of the two wheels:

<p style="text-align:center;"><img src="https://raw.githubusercontent.com/duckietown/duckietown-lx/0b59215ef7ff4ce7ae608b9546db3ef9a840d9d0/modcon/assets/images/odometry/odometry-baseline.png" width="300" alt="odometry-baseline"></p>  

If you have a robot, take a ruler and measure it!

In [ ]:
# What is the baseline length of your robot?

baseline_wheel2wheel = 0.1 #  Take a ruler and measure the distance between the center of the two wheels (meters)

Note: the default value, and that used in simulation, is $baseline = 0.1m$.

We are now ready to calculate the rotation of the Duckiebot:

In [ ]:
# Of what angle has the robot rotated?

Delta_Theta = # [radians]

print(f"The robot has rotated: {np.rad2deg(Delta_Theta)} degrees")

### 🚙 (Optional) Save your new value of `baseline`

Again, this isn't all that necessary unless your bot is way out of wack

Let's make sure it remembers its new wheel baseline!

Power you Duckiebot on, make sure it is connected to the network and you can ping it, then open a terminal **on your computer** and type:

    dts start_gui_tools ROBOTNAME
    
    rosparam set /ROBOTNAME/kinematics_node/baseline baseline-value
    
where `baseline-value` is the value of `baseline_wheel2wheel` you just measured (expressed in meters). You can then save it with:

    rosservice call /ROBOTNAME/kinematics_node/save_calibration
    
and finally verify that it has been saved by opening the `ROBOTNAME.yaml` file in your Dashboard > File Manager > Calibrations > Kinematics page.

---



# Wheel encoders tutorial

Encoders are sensors that convert (analog) angular position, or motion of a shaft, into a digital signal.

In Duckietown we use Hall effect encoders link to Wikipedia, which extract the incremental change in angular position of the wheels. Every time the shaft rotates of a certain set angle, i.e., the resolution of the encoder, it emits a pulse. We call these pulses "ticks".

We can use ticks from both wheels to measure the variation of the position of the Duckiebot while it moves.

In this activity we learn how to access the data coming from the wheel encoders of our Duckiebot (whether physical or simulated), and understand what each field means. We will use this data for later activities.


## Understanding wheel encoder measurements.

Let's look at the raw data that the wheel encoders produce.

Messages will look like this:

```
header:
  seq: 372
  stamp:
    secs: 1618436796
    nsecs:  55785179
  frame_id: "argo/left_wheel_axis"
data: 4
resolution: 135
type: 1
```

Let's look at what each field means:

  * `seq`: is an incremental identifier of the message. For each message received, it will increase by one.
  * `stamp`: the timestamp of the message. (Note: this field will be empty when looking at it through VNC)
  * `data`: is the cumulative count of ticks from the encoder in this instance. It will increase if the wheel is spinning forward, decrease if backwards. This is the actual measurement we can use to build our algorithms going forward.
  * `resolution`: is the total number of ticks for each full revolution of the wheel (a constant).
  * `type`: indicates the kind of encoder measurements. 1 stands for incremental measurements.

### 🚙 Read data from wheel encoders (on your Duckiebot)

1. Open a terminal on your computer and type:

     ```
     dts start_gui_tools ROBOTNAME
     ```

2. Then run:

     ```
     rostopic echo /ROBOTNAME/left_wheel_encoder_node/tick
     ```

3. Start up the keyboard control and command your robot to go forward, you will see the wheel encoder messages.

```
---
header:
  seq: 372
  stamp:
    secs: 1618436796
    nsecs:  55785179
  frame_id: "argo/left_wheel_axis"
data: 4
resolution: 135
type: 1
---
```

## Reading the number of ticks from each wheel

The wheel encoder message above provides several pieces of information. Let's extract data from it.



In [ ]:
# We define this function just to show how to extract data from the encoder_msg, which is the data coming from the wheel ecoders
# This code block won't produce anything.

def EncoderCallback(encoder_msg):

    N_tot = encoder_msg.resolution # number of ticks per wheel revolution
    ticks = encoder_msg.data # incremental count of ticks from the encoder

    #Let's see if we've done it right
    print("The received message is :")
    print()
    print(encoder_msg)
    print()
    print(f"N. of ticks : {ticks}")
    print(f"Total ticks : {N_tot}")

You should now be able to read and understand wheel encoder data, and extract fields of interest from the messages. You can proceed to the rest of the homework.